# Soccer Analytics — Run Everything

Runs on any GPU box (RunPod, Colab, local) — it auto-detects which.

**What this produces:** an annotated match clip with tracked players coloured by team,
a live tactical minimap, per-player distance covered, and possession percentages.

**How to read this notebook.** Every stage opens with an explanation of *why* it works
that way before the code that does it. The intent is that after running it you could
rebuild the pipeline from scratch, not just re-execute cells. The architecture and the
trade-offs behind each choice are in `walkthrough.html`; detector internals (anchors,
NMS, label assignment) are in `label_assignment.html`.

**Order matters** — sections 1→8 build on each other. Run them in order the first time.

## 0 · Setup

`imgsz=1280` rather than the usual 640 is a deliberate choice you'll see repeated: a
football is roughly 10 px in a 1080p frame, and downscaling to 640 shrinks it below
what any detector can reliably fire on.

In [ ]:
!pip install -q ultralytics "supervision>=0.29,<0.30" "transformers<5" opencv-python-headless "numpy<2.4" scikit-learn

In [ ]:
import json, shutil, subprocess, sys
from collections import defaultdict, deque
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
from ultralytics import YOLO

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO = Path.cwd()
sys.path.insert(0, str(REPO / "src"))
WORK = Path("/content/soccer_run") if IN_COLAB else REPO / "soccer_run"
WORK.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS = {}
print(f"device: {device} | colab: {IN_COLAB} | artifacts -> {WORK}")
if device != "cuda":
    print("WARNING: no GPU. Inference will be slow and fine-tuning impractical.")

## 1 · Get a clip

Point `SOURCE` at your own soccer footage — a 30–60 s broadcast clip is ideal. If you
haven't got one yet, the cell falls back to a street-scene clip from `supervision`'s
public assets, which exercises detection, tracking and clustering properly.

**Why not a sports clip as the stand-in?** The obvious choice was `VideoAssets.BASKETBALL`,
and it fails: measured on this exact pipeline it yields **0 player crops in the first 20
frames** (the camera is far back, so players are small and don't survive downscaling),
against **812** for the street clip. A stand-in that silently detects nothing is worse
than no stand-in. Note what the clustering then means here: it separates people by
*clothing colour*, not team — the mechanism is identical, the semantics aren't.

In [ ]:
SOURCE = None   # <-- put your clip path here, e.g. "match.mp4"

if SOURCE is None:
    from supervision.assets import VideoAssets, download_assets
    SOURCE = download_assets(VideoAssets.PEOPLE_WALKING)
    print(f"no clip supplied — using stand-in: {SOURCE}")

info = sv.VideoInfo.from_video_path(SOURCE)
print(f"{info.width}x{info.height} @ {info.fps}fps, {info.total_frames} frames")

first_frame = next(sv.get_video_frames_generator(SOURCE))
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
plt.title("frame 0 — you'll use this to place pitch keypoints in §5")
plt.axis("off"); plt.show()

## 2 · Detection, and the class-index trap

A detector looks at one frame in isolation and returns boxes. It has **no memory** —
nothing in its output says "this is the same player as last frame". That's §3's job,
and keeping the two concerns separate is what makes the pipeline debuggable.

### The trap

Each detection carries a *class index*, and **an index only means something relative to
the model that produced it**:

| index | COCO checkpoint | typical soccer checkpoint |
|---|---|---|
| 0 | `person` | **`ball`** |
| 1 | `bicycle` | `goalkeeper` |
| 2 | `car` | `player` |
| 3 | `motorcycle` | `referee` |

So `detections[class_id == 0]` means "people" on one model and "the ball" on the other.
Swap the checkpoint and your player tracker silently starts tracking a football —
and still produces a complete, plausible-looking set of statistics.

This is not hypothetical: the identical mistake invalidated the whole baseline
evaluation in the Traffic Lens project, where a COCO model was scored against a drone
dataset by index, so its `airplane` predictions were graded against *van* boxes.

**Rule: resolve classes by name, never by literal index.** `src/pipeline.py` does this
in `class_ids_for()`, and it fails loudly if nothing matches instead of quietly
detecting the wrong thing.

In [ ]:
from pipeline import class_ids_for, load_model, PLAYER_NAMES, BALL_NAMES, REFEREE_NAMES

model = load_model("yolo26n.pt")
print("this checkpoint's labels (first 8):",
      {i: model.names[i] for i in list(model.names)[:8]})
print("resolved players :", [model.names[i] for i in class_ids_for(model, PLAYER_NAMES)])
print("resolved ball    :", [model.names[i] for i in class_ids_for(model, BALL_NAMES)] or "— not in COCO-lite")
print("resolved referees:", [model.names[i] for i in class_ids_for(model, REFEREE_NAMES)] or "— needs a fine-tuned model")

In [ ]:
det = sv.Detections.from_ultralytics(model(first_frame, imgsz=1280, verbose=False)[0])
players = det[np.isin(det.class_id, class_ids_for(model, PLAYER_NAMES))]
print(f"{len(det)} objects, of which {len(players)} are players")

ann = sv.BoxAnnotator(thickness=2).annotate(first_frame.copy(), players)
plt.figure(figsize=(12, 7)); plt.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
plt.title(f"{len(players)} players detected at imgsz=1280"); plt.axis("off"); plt.show()

### 🔨 Try it
Re-run the cell above with `imgsz=640` and compare the count. On high-resolution
footage the difference is usually large, and it's entirely about how many pixels each
player survives the downscale with.

## 3 · Tracking — turning detections into identities

Detection gives boxes per frame. Tracking answers the harder question: **which box in
this frame is the same object as which box in the last frame?** Every derived statistic
depends on this — distance covered is meaningless if a player's identity changes
halfway through.

Two components do the work:

1. **A Kalman filter** predicts where each existing track *should* appear next, using a
   constant-velocity model. This matters more than it sounds: a fast-moving player's
   box may not overlap its own box in the next frame at all, so matching against the
   *last* position fails outright while matching against the *predicted* position works.
2. **The Hungarian algorithm** optimally matches those predictions to the new
   detections, scored by IoU.

**ByteTrack's contribution** is smaller and cleverer than it sounds: *don't throw away
low-confidence detections*. Conventional trackers discard everything below a threshold,
which is exactly wrong during occlusion — a partially hidden player still produces a
weak detection, and discarding it breaks the track. ByteTrack matches high-confidence
boxes first, then runs a second pass with the leftovers to rescue tracks that would
otherwise die.

The characteristic failure is the **ID switch**: two players cross and swap identities.
You can see it (a number jumping between people) but you can only *measure* it against
ground-truth tracking labels using MOTA/IDF1 — which this clip doesn't have, so we
report track length as a proxy and say so.

In [ ]:
tracker = sv.ByteTrack(frame_rate=info.fps)
seen = defaultdict(int)
for i, frame in enumerate(sv.get_video_frames_generator(SOURCE)):
    if i >= 60:
        break
    d = sv.Detections.from_ultralytics(model(frame, imgsz=1280, verbose=False)[0])
    d = tracker.update_with_detections(d[np.isin(d.class_id, class_ids_for(model, PLAYER_NAMES))])
    for t in d.tracker_id:
        seen[t] += 1

print(f"over 60 frames: {len(seen)} unique IDs, mean track length "
      f"{np.mean(list(seen.values())):.1f} frames")
print("short tracks (<10 frames) are usually fragmentation:",
      sum(1 for v in seen.values() if v < 10))

## 4 · Team assignment — unsupervised and free

Nobody labels which team each player is on. You don't need them to.

Kit colour dominates the appearance of a player crop, so if you embed each crop into a
vector space the crops form two obvious clusters. k-means with k=2 separates them in
milliseconds with no annotation and no training. `src/team_cluster.py` offers two
embedders:

- **HSV histogram** of the torso region (upper-centre of the crop, which is mostly
  shirt and avoids grass at the edges). Microseconds, CPU, works well when kits are
  colour-distinct.
- **SigLIP embedding** — a vision transformer's learned representation. ~100× slower,
  but robust to shadows and mixed floodlighting.

Start with the histogram. The point of trying the cheap thing first is that you find
out whether you need the expensive one.

**The honest failure:** goalkeepers wear a third kit and referees a fourth. With k=2
they are *forced* into a team — that's structural, not a tuning problem. Mitigations
are raising k and keeping the two largest clusters, or using a fine-tuned model that
detects `goalkeeper` and `referee` as their own classes.

Note how the crops are gathered: **scan until you have enough**, rather than reading a
fixed number of frames. A fixed count is fragile — a clip can open on a crowd shot, a
replay or an empty pitch, and you end up fitting k-means on an empty list. (That is
exactly what happened with the first stand-in clip chosen for this notebook, which
produced zero detections in its opening 20 frames.)

In [ ]:
from team_cluster import TeamClassifier
from pipeline import collect_crops

fit_crops = collect_crops(model, SOURCE, class_ids_for(model, PLAYER_NAMES),
                          target=200, max_scan=120, imgsz=1280)

teams = TeamClassifier("histogram").fit(fit_crops)
assign = teams.predict(fit_crops[:16])
print(f"fitted on {len(fit_crops)} crops; first 16 assignments: {assign}")

fig, axes = plt.subplots(2, 8, figsize=(15, 4.6))
for ax, crop, t in zip(axes.flat, fit_crops[:16], assign):
    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(f"team {t}", fontsize=9, color="tab:red" if t == 0 else "tab:blue")
    ax.axis("off")
plt.suptitle("crops grouped by cluster — check these actually look like two teams")
plt.tight_layout(); plt.show()

### 🔨 Look at that grid before continuing

**On real soccer footage** you should see a roughly even split, with the two groups
obviously corresponding to two kits. If they don't, nothing downstream is right —
common causes are a clip where one team barely appears, or floodlighting that flattens
the colour difference (try `TeamClassifier("siglip")`).

**On the street-scene stand-in, expect a lopsided split** — roughly 19:1 in testing.
That is not a bug. There are no uniforms, so k-means is splitting a continuous spread
of clothing colours at an arbitrary boundary rather than finding two real groups. It
demonstrates that the mechanism runs; it can't demonstrate that it's *correct*. Only
footage with two actual kits can do that, which is the main reason to swap in your own
clip before drawing conclusions from anything below.

## 5 · Homography — the geometry that makes it real

This is the heart of the project, and the only part with no learning involved.

**Pixel distances are meaningless.** Two players 10 m apart near the camera span
hundreds of pixels; the same 10 m at the far touchline spans a few dozen. Any statistic
computed in pixel space is not merely imprecise — it's wrong by a *different factor at
every depth*.

The fix rests on one fact: **a pitch is a plane.** Any two views of a plane are related
by a single 3×3 matrix, a homography. It has 8 degrees of freedom, so 4 point
correspondences suffice — each gives two equations.

### Why this project is luckier than the last one

**A pitch is 105 × 68 m by regulation.** The real-world coordinates aren't estimated,
they're specified by the laws of the game. Traffic Lens had no such reference: its
calibration quad was eyeballed at 30 m of road depth where the true span was ~250 m, so
every speed came out ~8× too low and the pipeline cheerfully reported 55 km/h traffic on
a free-flowing motorway. That entire class of error is off the table here — *if* you
place the keypoints on landmarks whose pitch coordinates you actually know.

### Placing keypoints

Read pixel coordinates off the frame printed in §1 for four landmarks you can identify,
and pair them with their known position in metres. Good choices: corner flags, penalty-box
corners, the halfway line meeting each touchline.

```
(0,0) ───────────── (52.5,0) ───────────── (105,0)
  │                     │                     │
  │      centre spot (52.5, 34)               │
  │                     │                     │
(0,68) ──────────── (52.5,68) ──────────── (105,68)
```

**Spread them out.** Four points clustered in one corner make the solve
ill-conditioned — you still get a matrix, it just isn't the right one, and the failure
is numerical rather than obvious.

In [ ]:
# EDIT THESE for your footage. Defaults are a rough guess for the stand-in clip and
# will NOT be correct for your video — the whole point of §5 is that these must be read
# off the actual frame.
keypoints = {
    "image_points": [[int(info.width * .10), int(info.height * .55)],
                     [int(info.width * .90), int(info.height * .55)],
                     [int(info.width * .98), int(info.height * .95)],
                     [int(info.width * .02), int(info.height * .95)]],
    "pitch_points": [[0, 0], [105, 0], [105, 68], [0, 68]],
}
(WORK / "keypoints.json").write_text(json.dumps(keypoints, indent=2))

from homography import PitchMapper, draw_minimap
mapper = PitchMapper(keypoints["image_points"], keypoints["pitch_points"])

prev = first_frame.copy()
cv2.polylines(prev, [np.array(keypoints["image_points"], np.int32)], True, (0, 0, 255), 4)
plt.figure(figsize=(12, 7)); plt.imshow(cv2.cvtColor(prev, cv2.COLOR_BGR2RGB))
plt.title("your calibration quad — does it sit on the four landmarks you claimed?")
plt.axis("off"); plt.show()

### Sanity-check the calibration before trusting anything downstream

Map a landmark you did **not** use for fitting and see how far off it lands. Under a
metre is good. This is the single cheapest check that separates "the pipeline ran" from
"the numbers mean something" — and it's exactly the check Traffic Lens lacked.

In [ ]:
probe_px = [int(info.width * .5), int(info.height * .75)]
probe_m = mapper.to_pitch(np.array([probe_px]))[0]
print(f"pixel {probe_px} -> pitch ({probe_m[0]:.1f}, {probe_m[1]:.1f}) m")
print("plausible?  x within 0-105, y within 0-68, and roughly where you'd expect.")
plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(draw_minimap(np.array([probe_m]), np.empty((0, 2))), cv2.COLOR_BGR2RGB))
plt.title("that probe point on the tactical map"); plt.axis("off"); plt.show()

## 6 · Full pipeline

Everything above, composed. Two implementation notes worth understanding:

**Two passes over the video.** k-means can't predict before it's fitted, and the kit
colours aren't known until players have been seen — so the clip is read once to fit and
once to render. The cost is double decode time. For a live stream you'd instead fit on
a rolling window and accept a provisional model for early frames.

**Team is cached per `tracker_id`.** A player's team doesn't change, so classify once
and reuse. This kills the per-frame re-embedding cost (critical with SigLIP) *and*
stops the assignment flickering between frames — two problems, one change.

In [ ]:
from pipeline import process_video

stats = process_video(
    source=SOURCE,
    output_path=str(WORK / "annotated.mp4"),
    model=model,
    keypoints=keypoints,
    embedder="histogram",
    imgsz=1280,
    max_frames=200,
    fit_frames=20,
)
RESULTS["pipeline"] = stats
print(json.dumps(stats, indent=2))

## 7 · Reading the stats critically

The arithmetic is easy. Whether the numbers *mean* anything is the hard part:

| Statistic | How it goes wrong |
|---|---|
| **Distance** | An ID switch teleports a player and adds tens of metres in one frame. The pipeline discards steps >2 m (180 km/h at 25fps — impossible), which makes distance a **lower bound**, not a measurement. |
| **Speed** | Per-frame differencing amplifies jitter; needs a window and a sanity check against plausible human speed (top sprinters ≈ 36 km/h). |
| **Possession** | Raw nearest-player-to-ball flickers every frame when two players contest. The pipeline requires a 60% majority over a half-second window. |
| **Team shape** | One misassigned goalkeeper drags a centroid the length of the pitch. |

Sanity anchors from outside the code, which is what makes them useful: a footballer
covers **~10–12 km in 90 minutes**. Extrapolate your clip and see whether it's in the
right order of magnitude. If a 20-second clip implies 40 km, something upstream is wrong.

In [ ]:
if stats["top_distance_m"]:
    secs = stats["frames"] / info.fps
    print(f"clip length: {secs:.1f}s\n")
    for row in stats["top_distance_m"]:
        per90 = row["metres"] / secs * 5400 / 1000
        flag = "plausible" if 5 < per90 < 16 else "IMPLAUSIBLE — check calibration"
        print(f"  #{row['id']:>3}: {row['metres']:6.1f} m  ->  {per90:5.1f} km/90min   {flag}")
else:
    print("no distance stats — keypoints missing or no tracks survived")

## 8 · Artifacts

In [ ]:
frames = list(sv.get_video_frames_generator(str(WORK / "annotated.mp4")))
if frames:
    plt.figure(figsize=(13, 8))
    plt.imshow(cv2.cvtColor(frames[min(60, len(frames) - 1)], cv2.COLOR_BGR2RGB))
    plt.title("annotated output — players by team, minimap bottom-left"); plt.axis("off"); plt.show()

media = WORK / "media"; media.mkdir(exist_ok=True)
subprocess.run(f'ffmpeg -y -v error -i "{WORK}/annotated.mp4" -vf '
               f'"fps=8,scale=640:-1:flags=lanczos,palettegen" "{media}/pal.png"', shell=True)
subprocess.run(f'ffmpeg -y -v error -i "{WORK}/annotated.mp4" -i "{media}/pal.png" -lavfi '
               f'"fps=8,scale=640:-1:flags=lanczos[x];[x][1:v]paletteuse" "{media}/demo.gif"', shell=True)
subprocess.run(f'ffmpeg -y -v error -ss 2 -i "{WORK}/annotated.mp4" -frames:v 1 '
               f'-vf scale=1400:-1 -q:v 3 "{media}/frame.jpg"', shell=True)
(media / "pal.png").unlink(missing_ok=True)

(WORK / "results.json").write_text(json.dumps(RESULTS, indent=2))
arts = [WORK / "results.json", WORK / "keypoints.json", media / "demo.gif", media / "frame.jpg"]
if IN_COLAB:
    for p in arts:
        if p.exists():
            files.download(str(p))
else:
    print(f"artifacts under {WORK}:")
    for p in arts:
        print(f"  {'OK ' if p.exists() else 'MISSING'} {p}")

## What's next

This runs end to end with a COCO-pretrained model, which finds *people* but not the
ball, goalkeepers or referees. To go further:

1. **Fine-tune a soccer detector** on the Roboflow football dataset (free API key,
   ~600 labelled broadcast frames with `ball / goalkeeper / player / referee`). Pass the
   checkpoint via `weights=` — everything else works unchanged, *because classes are
   resolved by name*.
2. **Expect the ball to be hard.** ~10 px, motion-blurred, occluded. Raise `imgsz`,
   try SAHI tiled inference, or add motion priors. Report its per-class mAP separately —
   never hidden inside an average.
3. **Your own footage**, with keypoints read off real landmarks.
4. **Learned pitch keypoints** so calibration works on any broadcast without clicking —
   the stretch goal, and what SoccerNet's calibration labels are for.